And we're done! If you have any questions on Unsloth, we have a [Discord](https://discord.gg/unsloth) channel! If you find any bugs or want to keep updated with the latest LLM stuff, or need help, join projects etc, feel free to join our Discord!

Some other resources:
1. Train your own reasoning model - Llama GRPO notebook [Free Colab](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Llama3.1_(8B)-GRPO.ipynb)
2. Saving finetunes to Ollama. [Free notebook](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Llama3_(8B)-Ollama.ipynb)
3. Llama 3.2 Vision finetuning - Radiography use case. [Free Colab](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Llama3.2_(11B)-Vision.ipynb)
4. See notebooks for DPO, ORPO, Continued pretraining, conversational finetuning and more on our [documentation](https://unsloth.ai/docs/get-started/unsloth-notebooks)!

<div class="align-center">
  <a href="https://unsloth.ai"><img src="https://github.com/unslothai/unsloth/raw/main/images/unsloth%20new%20logo.png" width="115"></a>
  <a href="https://discord.gg/unsloth"><img src="https://github.com/unslothai/unsloth/raw/main/images/Discord.png" width="145"></a>
  <a href="https://unsloth.ai/docs/"><img src="https://github.com/unslothai/unsloth/blob/main/images/documentation%20green%20button.png?raw=true" width="125"></a>

  Join Discord if you need help + ⭐️ <i>Star us on <a href="https://github.com/unslothai/unsloth">Github</a> </i> ⭐️
</div>

  This notebook and all Unsloth notebooks are licensed [LGPL-3.0](https://github.com/unslothai/notebooks?tab=LGPL-3.0-1-ov-file#readme).

# Task
Perform reasoning-based sentiment analysis on 80 headlines from the "dataset.xlsx" file using the `gpt-oss-20b` model. The model should first provide a step-by-step reasoning process, followed by a single sentiment label (Positive, Negative, or Neutral) for each headline. Display both the detailed reasoning and the predicted sentiment.

## Verify GPT-OSS Model Configuration

### Subtask:
Confirm that the `gpt-oss-20b` model is loaded and properly configured for reasoning, specifically by setting `reasoning_effort='high'` in the `tokenizer.apply_chat_template` function. This ensures the model engages its full reasoning capabilities as demonstrated in the notebook.


**Reasoning**:
The first instruction is to run the installation cell. This code block will install the necessary libraries for Unsloth.



In [ ]:
%%capture
import os, importlib.util
!pip install --upgrade -qqq uv
if importlib.util.find_spec("torch") is None or "COLAB_" in "".join(os.environ.keys()):
    try: import numpy, PIL; _numpy = f"numpy=={numpy.__version__}"; _pil = f"pillow=={PIL.__version__}"
    except: _numpy = "numpy"; _pil = "pillow"
    !uv pip install -qqq \
        "torch>=2.8.0" "triton>=3.4.0" {_numpy} {_pil} torchvision bitsandbytes "transformers==4.56.2" \
        "unsloth_zoo[base] @ git+https://github.com/unslothai/unsloth-zoo" \
        "unsloth[base] @ git+https://github.com/unslothai/unsloth" \
        git+https://github.com/triton-lang/triton.git@0add68262ab0a2e33b84524346cb27cbb2787356#subdirectory=python/triton_kernels
elif importlib.util.find_spec("unsloth") is None:
    !uv pip install -qqq unsloth
!uv pip install --upgrade --no-deps transformers==4.56.2 tokenizers trl==0.22.2 unsloth unsloth_zoo

**Reasoning**:
The next step is to load the `gpt-oss-20b` model and its tokenizer using `FastLanguageModel.from_pretrained` as per the subtask instructions.



In [ ]:
from unsloth import FastLanguageModel
import torch

# 4bit pre quantized models we support for 4x faster downloading + no OOMs.
fourbit_models = [
    "unsloth/gpt-oss-20b-unsloth-bnb-4bit", # 20B model using bitsandbytes 4bit quantization
    "unsloth/gpt-oss-120b-unsloth-bnb-4bit",
    "unsloth/gpt-oss-20b", # 20B model using MXFP4 format
    "unsloth/gpt-oss-120b",
] # More models at https://huggingface.co/unsloth

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/gpt-oss-20b",
    dtype = None, # None for auto detection
    max_seq_length = 4096, # Choose any for long context!
    load_in_4bit = False,  # 4 bit quantization to reduce memory
    full_finetuning = False, # [NEW!] We have full finetuning now!
    # token = "YOUR_HF_TOKEN", # HF Token for gated models
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.2.1: Fast Gpt_Oss patching. Transformers: 4.56.2.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: Using float16 precision for gpt_oss won't work! Using float32.
Unsloth: QLoRA and full finetuning all not selected. Switching to 16bit LoRA.


model.safetensors.index.json: 0.00B [00:00, ?B/s]

model-00000-of-00002.safetensors:   0%|          | 0.00/4.79G [00:00<?, ?B/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/4.80G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/4.17G [00:00<?, ?B/s]

...

generation_config.json:   0%|          | 0.00/165 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/27.9M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/446 [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

## Load Headlines from dataset.xlsx

### Subtask:
Load the headlines from the provided `dataset.xlsx` file. We will extract a small subset of 80 headlines from this dataset for sentiment analysis.


**Reasoning**:
First, I will import the pandas library, then load the `dataset.xlsx` file into a DataFrame, extract the 'Headline' column, select the first 80 headlines, and finally convert them into a list.



In [ ]:
import pandas as pd

# Load the dataset from the Excel file
df = pd.read_excel('/content/dataset.xlsx')

# Extract the 'Headline' column
headlines_series = df['Headline']

# Select the first 80 headlines
first_80_headlines = headlines_series.head(80)

# Store these 80 headlines in a list
headlines_list = first_80_headlines.tolist()

print(f"Successfully loaded {len(headlines_list)} headlines from 'dataset.xlsx'.")
print("First 5 headlines:")
for i, headline in enumerate(headlines_list[:5]):
    print(f"{i+1}. {headline}")

KeyError: 'Headline'

**Reasoning**:
The previous code failed because the 'Headline' column does not exist in the DataFrame. Based on the kernel state, the correct column name for the headlines is 'text'. I will update the code to use the 'text' column.



In [ ]:
import pandas as pd

# Load the dataset from the Excel file
df = pd.read_excel('/content/dataset.xlsx')

# Extract the 'text' column (corrected from 'Headline')
headlines_series = df['text']

# Select the first 80 headlines
first_80_headlines = headlines_series.head(80)

# Store these 80 headlines in a list
headlines_list = first_80_headlines.tolist()

print(f"Successfully loaded {len(headlines_list)} headlines from 'dataset.xlsx'.")
print("First 5 headlines:")
for i, headline in enumerate(headlines_list[:5]):
    print(f"{i+1}. {headline}")

Successfully loaded 80 headlines from 'dataset.xlsx'.
First 5 headlines:
1. According to Gran, the company has no plans to move all production to Russia, although that is where the company is growing.
2. Technopolis plans to develop in stages an area of no less than 100,000 square meters in order to host companies working in computer technologies and telecommunications, the statement said.
3. The international electronic industry company Elcoteq has laid off tens of employees from its Tallinn facility contrary to earlier layoffs the company contracted the ranks of its office workers, the daily Postimees reported.
4. With the new production plant the company would increase its capacity to meet the expected increase in demand and would improve the use of raw materials and therefore increase the production profitability.
5. According to the company's updated strategy for the years 2009-2012, Basware targets a long-term net sales growth in the range of 20%-40% with an operating profit marg

**Reasoning**:
Now that the headlines are loaded, I will prepare the model for inference. The model and tokenizer have already been loaded, and the prompt construction and inference process for a single input with `reasoning_effort='high'` has been demonstrated in the notebook. I will adapt this to iterate through the `headlines_list`, generating a prompt for each, and then performing inference to get the reasoning and sentiment. I will use a `max_new_tokens` of 2048 to allow for detailed reasoning.



In [ ]:
from transformers import TextStreamer

sentiment_analysis_results = []

for i, headline in enumerate(headlines_list):
    print(f"\n--- Processing Headline {i+1}/{len(headlines_list)} ---")
    messages = [
        {"role": "system", "content": "You are an expert sentiment analysis AI. Analyze the given headline and determine if its sentiment is Positive, Negative, or Neutral. Provide a step-by-step reasoning process before giving the final sentiment label."},
        {"role": "user", "content": f"Analyze the sentiment of the following headline: '{headline}'"},
    ]

    inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt = True,
        return_tensors = "pt",
        return_dict = True,
        reasoning_effort = "high",
    ).to("cuda")

    print("\nModel Response:")
    streamer = TextStreamer(tokenizer, skip_prompt=True, skip_special_tokens=True)
    _ = model.generate(**inputs, max_new_tokens = 2048, streamer = streamer)

    # In a real scenario, you would parse the streamer output to extract reasoning and sentiment
    # For this task, we will just print the streamed output as requested.



--- Processing Headline 1/80 ---

Model Response:


analysisThe user: "Analyze the sentiment of the following headline: 'According to Gran, the company has no plans to move all production to Russia, although that is where the company is growing.'"

We need to determine if sentiment is Positive, Negative, Neutral. Provide step-by-step reasoning. We must analyze the sentiment of the headline. The headline: "According to Gran, the company has no plans to move all production to Russia, although that is where the company is growing." It says company has no plans to move all production to Russia, but that is where it is growing. So it's reporting some information but no overt positive or negative language. The statement "no plans to move all production" is somewhat neutral or maybe slightly negative for those concerned with Russian involvement. "although that is where the company is growing" is positive regarding growth but not necessarily positive sentiment globally. The sentiment is more informational. So leaning to Neutral. There are no st


KeyboardInterrupt

